# Part 2 of DQN: Implementing DQN from sratch using Deep Mind papper

In [11]:

from environment import WumpusWorldEnvironment
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
import torch.nn as nn
#we use this library to apply operations sch as one-hot encoding to the tensore
import torch.nn.functional as F

### For simplicity we will use the interger (0-35) vector to represent the environemnt to keep it concise. The input to the nueral network is a vector of length 36 each hidden layer has 64 nuerons and the output has 4 units. No need to use the CNN unless we use images, Future iterations of the project can include multiple.

In [2]:
#First we will provide base hyperparamters(these can be altered for hyperatamter tuning)
BATCH_SIZE = 64
REPLAY_BUFFER_SIZE = 150000
GAMMA = 0.99
LEARNING_RATE = 0.0001
EPSILON_START = 1.0
EPSILON_END = 0.03
EPSILON_DECAY_STEPS = 200
MAX_EPISODES = 1000
TARGET_UPDATE = 150
#use macbook gpu.
device = torch.device("mps")


In [3]:
#Defining the Q_network structure. We are using the (0-35) vector represneation
class Q_Network(nn.Module):
    """Maps one-hot gird posiiton to Q_Values for the discrete action set:"""
    def __init__(self):
        super().__init__()
        self.neural_network_vector2  = nn.Sequential(
        nn.Linear(36,64),
        nn.ReLU(),
        nn.Linear(64,4)
    )

    #Forward propogation
    def forward(self,state):
        return self.neural_network_vector2(state)

In [4]:
"""Converting the envionements integer psoition to a tensor"""
def encode_state(state):
    state_tensor = torch.tensor(int(state),dtype=torch.long,device=device)
    return F.one_hot(state_tensor,num_classes=36).float().unsqueeze(0)


In [ ]:
"""INITILAIZING THE REPLAY MEMORY BUFFER WITh entires: (s,a,r,s',termination_flags)"""
#we use a queue to represent the replay buffer so the first episode we saw gets dequeed first in constant time.,
replay_memory = deque(maxlen=REPLAY_BUFFER_SIZE)
#initaize the live q_network and the target q_netwrk used to compute MSE
online_network = Q_Network().to(device)
target_network = Q_Network().to(device)
target_network.load_state_dict(online_network.state_dict())
target_network.eval()

#initilaize the loss function
optimizer = torch.optim.Adam(online_network.parameters(),lr=LEARNING_RATE)
loss_function = nn.MSELoss()

#initlaize the wumpus environemnt and initial values
environment = WumpusWorldEnvironment(observation_type='integer', action_type='discrete')
env_steps = 0
optimizer_steps = 0
results = []

In [9]:
"""Picking an action given a state via epislon greedy and returning the assoicate Q_value via the bellman equation"""
def pick_action(state,epsilon):
    #we choose to explore
    if random.random() < epsilon:
        return environment.action_space.sample()

    #we choose to take the greeyd action(action with the highest q_value)
    with torch.no_grad():
        #run the online nueral network to get q_values
        q_values = online_network(encode_state(state))
    return int(q_values.argmax(dim=1).item())

In [10]:
"""Now we implemnt the traning/Q_network update loop for the DQN algorithm. following the following steps
    1. Sample mini Batch


"""
def train():
    #Sample a mini_batch and make a gradient descent update
    global optimizer_steps
    if len(replay_memory) < BATCH_SIZE:
        return None

    mini_batch = random.sample(replay_memory,BATCH_SIZE)
    states,action,rewards,next_states,termination = zip(*mini_batch)
    #one-hot encode the states and convert actions,and rweads to tensors
    states = F.one(torch.tensor(states, dtype=torch.long,device=device), num_classes=36).float()
    actions = torch.tensor(actions,dtype=torch.long,device=device).unsqueeze(1)
    rewards = torch.tensor(rewards,dtype=torch.float32,device=device)
    next_states = F.one(torch.tensor(next_states, dtype=torch.long,device=device), num_classes=36).float()
    termination = torch.tensor(termination,dtype=torch.long,device=device)

    #Get the predicted  values from the live Q network. Q_online(s,a)
    predictions = online_network(states).gather(1,actions).squeeze(1)
    #update the target network
    with torch.no_grad():
        next_q_values = target_network(next_states).max(dim=1).values
        targets = rewards + GAMMA * (1.0 - termination) * next_q_values

    #compute the loss, take rthe gradient and perform grdient descent/backpropagation
    loss = loss_function(predictions,targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    optimizer_steps+=1
    if optimizer_steps % TARGET_UPDATE == 0:
        target_network.load_state_dict(online_network.state_dict())

    return loss.item()





In [12]:
"""Now we finnally combine all the previous steps to conduct the DQN algorihtm for the environment reading the values produced."""
for episode in range(MAX_EPISODES):
    state,info = environment.reset()
    episode_return = 0.0
    losses = []

    for t in range(environment.max_timesteps):
        #to slighlt change epislon as training goes on so we are not stuck exploring
        epsilon = EPSILON_END + (EPSILON_START - EPSILON_END) * (max(0, EPSILON_DECAY_STEPS - env_steps) / EPSILON_DECAY_STEPS)
        action = pick_action(state,epsilon)
        next_state,reward,terminated,truncated,info = environment.step(action)
        done = terminated or truncated
        replay_memory.append((int(state),int(action),float(reward),int(next_state),done))

        loss = train()
        if loss is not None:
            losses.append(loss)

        state = next_state
        episode_return += reward
        environment_steps += 1
        #if we have terminated/ hit the the terminated state we are done
        if done:
            break

        results.append(episode_return)
        losses.append(float(np.mean(losses)) if losses else np.nan)
    

        


            


TypeError: float() argument must be a string or a real number, not 'NoneType'